# <B>TASK 3: SYSTEM INTEGRATION</B>

### Notes:
- The references we used in this `.ipynb` notebook are exclusively to support the technical development of our work (such as library usage and function design), and do not constitute the formal references cited in our final report.
- Please install necessary libraries as needed.

In [10]:
# Import necessary library

import pandas as pd
import ipywidgets as widgets
import numpy as np
import time
import tracemalloc
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from sklearn.metrics import ndcg_score
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

### Utility function

The functions below are used in both <b>Task 1 (Pattern mining)</b> and <b>Task 2 (Collaborative Filtering)</b>. For the purpose of clean code and Separation of Concerns (SoC), we decided to modularise these functions into this section. This approach can be considerd as developling internal APIs, each encapsulating specific functionality to be triggered as needed for specific tasks (Martin, 2008).

In [11]:
# Declare function to generate item-based collaborative filtering recommendations using a K-Nearest Neighbors (KNN) model
def generate_recommendations(user_id, matrix, model, top_n=5):
    """
    Generates item-based collaborative filtering recommendations using a KNN model trained on item-user matrix data.

    Parameters:
        user_id (int/str): Identifier of the user for whom recommendations are generated.
        matrix (pd.DataFrame): Item-user interaction matrix with items as rows and users as columns.
        model (KNN): Pre-fitted K-Nearest Neighbors model to compute item similarity.
        top_n (int): Number of top recommended items to return (default is 5).

    Returns:
        list: Ranked list of item names most similar to the user’s past interactions, excluding previously seen items.
    """
    
    if user_id not in matrix.columns:
        return []

    past_items = matrix.index[matrix.loc[:, user_id] > 0].tolist()
    recommended_items = {}

    for item in past_items:
        item_vector = matrix.loc[item].values.reshape(1, -1)
        distances, indices = model.kneighbors(item_vector, n_neighbors=6)

        for i in range(1, len(indices.flatten())):  
            similar_item = matrix.index[indices.flatten()[i]]
            similarity_score = 1 - distances.flatten()[i]
            if similar_item in past_items:      
                continue
            recommended_items[similar_item] = recommended_items.get(similar_item, 0) + similarity_score

    sorted_recs = sorted(recommended_items.items(), key=lambda x: x[1], reverse=True)
    return [item for item in sorted_recs[:top_n]]

# Declare function to build item-user interaction matrix and train a KNN model for item-based collaborative filtering
def build_cf_model(train_data):
    """
    Builds the item-user matrix and trains a KNN model.
    
    Parameters:
        train_data (pd.DataFrame): Transactional data containing 'itemDescription', 'User_id', and 'Transaction_ID' columns.
    
    Returns:
        item_user_matrix: DataFrame
        knn_model: Trained NearestNeighbors model
    """
    item_user_matrix = train_data.pivot_table(
        index='itemDescription',
        columns='User_id',
        values='Transaction_ID',
        aggfunc='count',
        fill_value=0
    )

    knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
    knn_model.fit(item_user_matrix)

    return item_user_matrix, knn_model

## <b>1. Task 1 - Pattern mining</b>

Implementation of the FP-growth algorithm (generate_top_items()) to mine frequent itemsets, incorporating recency scoring to prioritize recent transactions.

In [12]:
# --- Task 1: Pattern Mining ---
# (Pandas development team 2020; Scikit-learn developers 2023; Python Software Foundation 2023)

# Declare function to mine global frequent itemsets just once to reduce computation
def mine_frequent_itemsets(df):
    """
    Processes transactional data to identify globally frequent itemsets using the FP-Growth algorithm,
    enhanced with recency scoring to prioritize recent purchasing patterns.
    
    Parameters:
        df (pd.DataFrame): Input dataframe containing at least the following columns:
        User_id (int/str): Identifier of the user making the transaction.
        itemDescription (str): Description of items purchased.
        Date (str/date): Date of transaction (expects day-first format).

    Returns:
        list: Sorted list of unique items extracted from the top 5 most recent frequent itemsets.
    """
    df = df.dropna(subset=['User_id', 'itemDescription']).copy()
    df['User_id'] = df['User_id'].astype(int)
    df['itemDescription'] = df['itemDescription'].astype(str)
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
    df['Transaction_ID'] = df['User_id'].astype(str) + '_' + df['Date'].astype(str)
    df_sorted = df.sort_values('Date')
    split_index = int(0.7 * len(df_sorted))
    train_data = df_sorted.iloc[:split_index].copy()
    train_data = df_sorted

    train_data = train_data.sort_values(by=['User_id', 'Date'], ascending=[True, False])
    train_data['RecencyRank'] = train_data.groupby('User_id')['Date'].rank(method='dense', ascending=False)
    train_data['RecencyWeight'] = 1 / train_data['RecencyRank']
    recency_scores = train_data.groupby('itemDescription')['RecencyWeight'].sum().to_dict()

    baskets = train_data.groupby('Transaction_ID')['itemDescription'].apply(list).tolist()
    encoder = TransactionEncoder()
    encoded_baskets = encoder.fit_transform(baskets)
    encoded_df = pd.DataFrame(encoded_baskets, columns=encoder.columns_)

    frequent_itemsets = fpgrowth(encoded_df, min_support=0.005, use_colnames=True)
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
    frequent_itemsets = frequent_itemsets[frequent_itemsets['length'] > 1].copy()

    frequent_itemsets['recency_score'] = frequent_itemsets['itemsets'].apply(
        lambda items: sum([recency_scores.get(item, 0) for item in items])
    )
   
    top_itemsets = frequent_itemsets.sort_values(by='recency_score', ascending=False).head(5)['itemsets'].tolist()

    unique_items = sorted(set(item for itemset in top_itemsets for item in itemset))

    return unique_items

## <b>2. Task 2 - Collaborative Filtering</b>

Implements an item-based collaborative filtering approach using K-Nearest Neighbors (KNN) with a cosine similarity metric

In [13]:
# --- Task 2: Collaborative Filtering ---

# (Pandas development team 2020; Scikit-learn developers 2023)

def colab_filtering(user_id, item_user_matrix, knn_model, top_n=5):
    """
    Generate top-N item recommendations for a user using collaborative filtering.

    Parameters:
        user_id (int): The user to generate recommendations for.
        item_user_matrix (DataFrame): Item-user interaction matrix.
        knn_model (NearestNeighbors): Pretrained KNN model.
        top_n (int): Number of recommendations.

    Returns:
        List of recommended itemDescriptions.
    """
    if user_id not in item_user_matrix.columns:
        print(f'User {user_id} not found in training data.')
        return []

    user_items = item_user_matrix.index[item_user_matrix[user_id] > 0].tolist()
    recommendations = {}

    for item in user_items:
        item_vector = item_user_matrix.loc[item].values.reshape(1, -1)
        distances, indices = knn_model.kneighbors(item_vector, n_neighbors=min(6, len(item_user_matrix)))

        for i in range(1, len(indices.flatten())):
            similar_item = item_user_matrix.index[indices.flatten()[i]]
            similarity_score = 1 - distances.flatten()[i]

            if similar_item in user_items:
                continue

            recommendations[similar_item] = recommendations.get(similar_item, 0) + similarity_score

    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in sorted_recs[:top_n]]


## <b>3. Task 3 - System integration</b>

In [19]:
# --- Task 3: System Integration ---

# (Pandas development team 2020; Jupyter Widgets Team 2023; Python Software Foundation 2023)

# Create widgets for user input
user_id_input = widgets.Text(description='User ID:', placeholder="Enter User ID or 'exit'")
option_input = widgets.Dropdown(options=['with', 'without'], description='Method:')
submit_button = widgets.Button(description='Get Recommendations', layout=widgets.Layout(width='200px'))
output = widgets.Output()

try:
    submit_button._click_handlers.callbacks.clear()
except:
    pass


def on_submit_button_clicked(b):
    with output:
        output.clear_output()
        recommendation_system(user_id_input.value, option_input.value)


submit_button.on_click(on_submit_button_clicked)


display(user_id_input, option_input, submit_button, output)

# Declare function to handle cold-start users by recommending globally popular items from the training set
def check_users(train_df, user_id):
    """
    Handles cold-start scenario by recommending popular items to users with no prior purchase history.

    Parameters:
        train_df (pd.DataFrame): Training dataset containing at least the 'itemDescription' column.
        user_id (int or str): Identifier of the user to check in the training data.
    """
    print(f'User {user_id} has no purchase history in the training data.')
    print('Falling back to popular items (based on global frequency).')
    popular_items = train_df['itemDescription'].value_counts().head(5).index.tolist()
    print(f'Top 5 Popular Items: {popular_items}')
    
# Declare function to augment collaborative filtering results with unseen globally frequent items for better personalization
def augment_cf_recommendation(user_id, train_df, frequent_items, item_user_matrix, knn_model, top_n=5):
    """
    Combine collaborative filtering recommendations with frequent itemset patterns, boosting unseen frequent items.
    
    Parameters:
        user_id (int): User ID
        train_df (DataFrame): Full training dataframe
        frequent_items (list): Global frequent item list (output from mine_frequent_itemsets)
        item_user_matrix (DataFrame): Output of build_cf_model
        knn_model (NearestNeighbors): Trained model on item-user matrix
        top_n (int): Total number of recommendations to return
    
    Returns:
        List[str]: Final recommendations
    """
    cf_recs = colab_filtering(user_id, item_user_matrix, knn_model, top_n=top_n)
    user_history = set(train_df[train_df['User_id'] == user_id]['itemDescription'].unique())
    pattern_set = set(frequent_items) - user_history
    cf_set = set(cf_recs)

    boosted = list(cf_set & pattern_set)
    rest = list((cf_set | pattern_set) - set(boosted))

    recommendations = boosted + rest
    return recommendations[:top_n]

# Recommendation system main function
def recommendation_system(user_input, option):
    train_df = pd.read_csv('./Groceries data train.csv')
    train_df = train_df.dropna(subset=['User_id', 'itemDescription']).copy()
    train_df = train_df.astype({'User_id':'int', 'year':'int', 'month': 'int', 'day':'int', 'day_of_week':'int'})
    train_df['User_id'] = train_df['User_id'].astype(int)
    train_df['Date'] = pd.to_datetime(train_df['Date'], dayfirst=True)
    
    if 'Time' in train_df.columns:
        train_df['Timestamp'] = train_df['Date'].astype(str) + ' ' + train_df['Time'].astype(str)
        train_df['Transaction_ID'] = train_df['User_id'].astype(str) + '_' + train_df['Timestamp']
    else:
        train_df['txn_count'] = train_df.groupby(['User_id', 'Date']).cumcount()
        train_df['Transaction_ID'] = train_df['User_id'].astype(str) + '_' + train_df['Date'].astype(str) + '_' + train_df['txn_count'].astype(str)

    user_input = user_input.strip().lower()
    if user_input == 'exit':
        print('Exiting the system. Goodbye!')
        return

    try:
        user_id = int(user_input)
    except ValueError:
        print("Please enter a valid User ID (integer) or type 'exit' to quit.")
        return

    if option not in ['with', 'without']:
        print("Invalid option. Please choose 'with' or 'without'.")
        return

    if user_id not in train_df['User_id'].values:
        check_users(train_df, user_id)
        return

    item_user_matrix, knn_model = build_cf_model(train_df)
    frequent_items = mine_frequent_itemsets(train_df)
    if option == 'with':
        print(f'Generating recommendations for User {user_id} using frequent itemsets...')
        # Generate frequent items using FP-growth
        recommendations = augment_cf_recommendation(user_id,train_df,frequent_items, item_user_matrix,knn_model,top_n=5)
        recommendations = recommendations[:5]
    else:
        print(f'Generating recommendations for User {user_id} without frequent itemsets...')
        recommendations = colab_filtering(user_id, item_user_matrix, knn_model, top_n=5)

    if recommendations:
        print(f'\nTop 5 Recommendations for User {user_id}:')
        for i, item in enumerate(recommendations, 1):
            
            print(f'{i}. {item}')
    else:
        print(f'No recommendations could be generated for User {user_id}.')

Text(value='', description='User ID:', placeholder="Enter User ID or 'exit'")

Dropdown(description='Method:', options=('with', 'without'), value='with')

Button(description='Get Recommendations', layout=Layout(width='200px'), style=ButtonStyle())

Output()

## <b>4. Evaluation</b>

In [15]:
test_df = pd.read_csv('./Groceries data test.csv')
train_df = pd.read_csv('./Groceries data train.csv')

train_df.dropna(inplace = True), test_df.dropna(inplace = True)

(None, None)

In [16]:
test_df.rename(columns={'user_id': 'User_id'}, inplace = True)
print(test_df.columns.tolist())

'User_id' in test_df.columns.tolist()

['User_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']


True

In [17]:
# Evaluation code taken from Task 2: Declare function to compute Precision, Recall, Hit Rate, and NDCG@k for a single user's recommendation result
def precision_recall_hit_ndcg(predicted, actual, k=5, all_items=None):
    """
    Computes evaluation metrics for a single user’s recommendation result.

    Parameters:
        predicted (list): Ranked list of recommended items.
        actual (list): List of ground truth (actually interacted) items.
        k (int): Cutoff rank for evaluation (default is 5).
        all_items (list, optional): Full list of items across dataset for NDCG calculation.

    Returns:
        tuple: (precision@k, recall@k, hit rate@k, ndcg@k)
    """
    predicted_top_k = predicted[:k]
    actual_set = set(actual)

    hits = [item in actual_set for item in predicted_top_k]
    precision = sum(hits) / k
    recall = sum(hits) / len(actual_set) if actual_set else 0
    hit_rate = int(any(hits))

    if all_items is None:
        all_items = list(set(predicted + actual))

    item_index = {item: idx for idx, item in enumerate(all_items)}
    y_true = np.zeros((1, len(all_items)))
    y_score = np.zeros((1, len(all_items)))

    for item in actual:
        if item in item_index:
            y_true[0, item_index[item]] = 1
    for rank, item in enumerate(predicted_top_k):
        if item in item_index:
            y_score[0, item_index[item]] = 1 / (rank + 1)

    ndcg = ndcg_score(y_true, y_score, k=k)
    return precision, recall, hit_rate, ndcg

# Declare function to evaluate collaborative filtering (with or without augmentation) across all test users
def evaluate_all_users(train_path, test_path, method='with', k=5):
    """
    Evaluates recommendation system performance across all users in the test set, 
    and records execution time and memory usage.

    Parameters:
        train_path (str): Path to the training dataset (CSV format).
        test_path (str): Path to the test dataset (CSV format).
        method (str): Recommendation method to use: 'with' for augmented CF, 'without' for pure CF.
        k (int): Number of top recommendations to evaluate (default is 5).

    Returns:
        dict: Averaged performance metrics across all test users, including:
              - avg_precision
              - avg_recall
              - avg_hit_rate
              - avg_ndcg
              - runtime_seconds
              - peak_memory_mb
    """
    start_time = time.time()
    tracemalloc.start()

    # Load and preprocess train and test datasets
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    test_df.rename(columns={'user_id': 'User_id'}, inplace=True)

    train_df = train_df.dropna(subset=['User_id', 'itemDescription']).copy()
    train_df = train_df.astype({'User_id': 'int', 'year': 'int', 'month': 'int', 'day': 'int', 'day_of_week': 'int'})
    train_df['Date'] = pd.to_datetime(train_df['Date'], dayfirst=True)

    if 'Time' in train_df.columns:
        train_df['Timestamp'] = train_df['Date'].astype(str) + ' ' + train_df['Time'].astype(str)
        train_df['Transaction_ID'] = train_df['User_id'].astype(str) + '_' + train_df['Timestamp']
    else:
        train_df['txn_count'] = train_df.groupby(['User_id', 'Date']).cumcount()
        train_df['Transaction_ID'] = train_df['User_id'].astype(str) + '_' + train_df['Date'].astype(str) + '_' + train_df['txn_count'].astype(str)

    test_df = test_df.dropna(subset=['User_id', 'itemDescription']).copy()
    test_df = test_df.astype({'User_id': 'int', 'year': 'int', 'month': 'int', 'day': 'int', 'day_of_week': 'int'})
    test_df['Date'] = pd.to_datetime(test_df['Date'], dayfirst=True)

    all_items = list(set(train_df['itemDescription']).union(set(test_df['itemDescription'])))
    metrics = []

    item_user_matrix, knn_model = build_cf_model(train_df)
    frequent_items = mine_frequent_itemsets(train_df)

    for user_id in tqdm(train_df['User_id'].unique(), desc='Evaluating Users'):
        if user_id not in test_df['User_id'].values:
            continue

        actual = test_df[test_df['User_id'] == user_id]['itemDescription'].unique().tolist()
        if not actual:
            continue

        try:
            if method == 'with':
                predicted = augment_cf_recommendation(user_id, train_df, frequent_items, item_user_matrix, knn_model, top_n=k)
            else:
                predicted = colab_filtering(user_id, item_user_matrix, knn_model, top_n=k)
        except Exception as e:
            print(f'Error for user {user_id}: {e}')
            continue

        if not predicted:
            continue

        metrics.append(precision_recall_hit_ndcg(predicted, actual, k=k, all_items=all_items))

    avg_precision = round(np.mean([m[0] for m in metrics]), 4)
    avg_recall = round(np.mean([m[1] for m in metrics]), 4)
    avg_hit_rate = round(np.mean([m[2] for m in metrics]), 4)
    avg_ndcg = round(np.mean([m[3] for m in metrics]), 4)

    # End timing and memory tracking
    runtime_seconds = round(time.time() - start_time, 2)
    current, peak = tracemalloc.get_traced_memory()
    peak_memory_mb = round(peak / 1024 / 1024, 2) 
    tracemalloc.stop()

    return {
        'avg_precision': avg_precision,
        'avg_recall': avg_recall,
        'avg_hit_rate': avg_hit_rate,
        'avg_ndcg': avg_ndcg,
        'runtime_seconds': runtime_seconds,
        'peak_memory_mb': peak_memory_mb
    }

result_with = evaluate_all_users('./Groceries data train.csv', './Groceries data test.csv', method='with')
print("'With' case metrics result:", result_with)

result_without = evaluate_all_users('./Groceries data train.csv', './Groceries data test.csv', method='without')
print("'Without' case metrics result:", result_without)

Evaluating Users: 100%|██████████| 3493/3493 [01:40<00:00, 34.63it/s]


'With' case metrics result: {'avg_precision': 0.204, 'avg_recall': 0.1911, 'avg_hit_rate': 0.6529, 'avg_ndcg': 0.2363, 'runtime_seconds': 102.05, 'peak_memory_mb': 26.43}


Evaluating Users: 100%|██████████| 3493/3493 [01:28<00:00, 39.45it/s]

'Without' case metrics result: {'avg_precision': 0.2014, 'avg_recall': 0.1886, 'avg_hit_rate': 0.6476, 'avg_ndcg': 0.248, 'runtime_seconds': 89.66, 'peak_memory_mb': 26.48}


## <b>Reference</b>

1. The pandas development team 2020, pandas-dev/pandas: Pandas version 1.1.3, Zenodo, viewed 14 April 2025, https://doi.org/10.5281/zenodo.3509134.

2. Scikit-learn developers 2023, scikit-learn: Machine Learning in Python version 1.3.0, viewed 14 April 2025, https://scikit-learn.org/stable/.

3. Python Software Foundation 2023, The Python Standard Library version 3.11, viewed 14 April 2025, https://docs.python.org/3/library/.

4. Jupyter Widgets Team 2023, ipywidgets: Interactive HTML widgets for Jupyter notebooks and the IPython kernel version 8.0, viewed 14 April 2025, https://github.com/jupyter-widgets/ipywidgets.

5. Martin, RC 2008, Clean code: a handbook of agile software craftsmanship, 1st edn, Prentice Hall, Upper Saddle River, NJ.